# Waterfall Plot — Pressão Cíclica de Combustão (H₂)

Visualização 3D interativa da pressão in-cylinder ciclo a ciclo,  
em função do ângulo do virabrequim (graus aTDC).  
Colormap: azul → pressão baixa, vermelho → pressão alta (knock).

In [1]:
# ── Dependências ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from pathlib import Path

# ── Caminhos ──────────────────────────────────────────────────────────────────
# Notebook em: DOUTORADO/Notebooks/  |  Dados em: DOUTORADO/Dados/
DATA_PATH   = Path("../Dados/P_H2.csv")
HTML_OUTPUT = Path("waterfall.html")

CSV_SEP      = ";"
CSV_ENCODING = "cp850"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {DATA_PATH.resolve()}")

print(f"Dados: {DATA_PATH.resolve()}")

Dados: C:\Users\GSA\Workspace\ALGORITMOS_IA\CURSO_IA\DOUTORADO\Dados\P_H2.csv


In [2]:
# ── 1. Ler cabeçalho e extrair números dos ciclos ─────────────────────────────
with open(DATA_PATH, encoding=CSV_ENCODING) as f:
    linha_ciclos  = f.readline().strip()   # Pressure Raw Data\MP ->;74;75;...
    linha_nomes   = f.readline().strip()   # Crank Angle;Praw;Praw;...
    linha_unidades = f.readline().strip()  # [deg aTDCf];[bar];[bar];...

cycle_nums = [
    c.strip()
    for c in linha_ciclos.split(CSV_SEP)[1:]   # pula o rótulo inicial
    if c.strip()
]
print(f"Ciclos encontrados : {len(cycle_nums)}  (MP {cycle_nums[0]} a MP {cycle_nums[-1]})")

# ── 2. Carregar dados ─────────────────────────────────────────────────────────
df_raw = pd.read_csv(
    DATA_PATH,
    sep=CSV_SEP,
    encoding=CSV_ENCODING,
    skiprows=3,
    header=None,
    on_bad_lines="skip",
)
df_raw = df_raw.dropna(axis=1, how="all")

# Nomes das colunas: Crank_Angle + Cycle_74 ... Cycle_165
n_data_cols = len(df_raw.columns)
col_names   = ["Crank_Angle"] + [f"Cycle_{c}" for c in cycle_nums]
df_raw.columns = col_names[:n_data_cols]
df_raw = df_raw.apply(pd.to_numeric, errors="coerce")

print(f"Shape              : {df_raw.shape[0]} linhas × {df_raw.shape[1]} colunas")
print(f"Ângulo (°aTDC)     : {df_raw['Crank_Angle'].min():.1f}° a {df_raw['Crank_Angle'].max():.1f}°")
df_raw.head()

Ciclos encontrados : 93  (MP 74 a MP 166)
Shape              : 7200 linhas × 94 colunas
Ângulo (°aTDC)     : -360.0° a 359.9°


,Crank_Angle,Cycle_74,Cycle_75,Cycle_76,Cycle_77,Cycle_78,Cycle_79,Cycle_80,Cycle_81,Cycle_82,...,Cycle_157,Cycle_158,Cycle_159,Cycle_160,Cycle_161,Cycle_162,Cycle_163,Cycle_164,Cycle_165,Cycle_166
0,-360.000000,1.511148,1.201457,1.186087,1.175746,1.17237,1.148788,1.172791,1.138285,1.181303,...,1.403879,1.399723,1.363675,1.361363,1.374694,1.399047,1.815694,1.399273,1.417483,1.408209
1,-359.899994,1.511148,1.189457,1.192087,1.157746,1.17837,1.154788,1.148791,1.126285,1.175303,...,1.379879,1.387723,1.351675,1.361363,1.374694,1.399047,1.815694,1.381273,1.417483,1.426209
2,-359.799988,1.511148,1.171457,1.198087,1.151746,1.19037,1.160788,1.154791,1.138285,1.163303,...,1.361879,1.369723,1.369675,1.373363,1.374694,1.393047,1.815694,1.387273,1.429483,1.432209
3,-359.700012,1.511148,1.171457,1.192087,1.169746,1.19637,1.154788,1.178791,1.156285,1.163303,...,1.367879,1.351723,1.387675,1.385363,1.380694,1.381047,1.815694,1.393273,1.429483,1.432209
4,-359.600006,1.505148,1.183457,1.186087,1.175746,1.18437,1.148788,1.184791,1.156285,1.175303,...,1.379879,1.357723,1.381675,1.379363,1.374694,1.381047,1.821694,1.399273,1.417483,1.420209


In [3]:
# ── 3. Preparar dados para o waterfall ────────────────────────────────────────

# Janela de combustão (região de interesse)
ANGLE_MIN, ANGLE_MAX = -30.0, 90.0
# Número máximo de ciclos exibidos no waterfall
MAX_CYCLES = 20

# Filtrar janela de ângulo
mask = df_raw["Crank_Angle"].between(ANGLE_MIN, ANGLE_MAX)
df = df_raw.loc[mask].reset_index(drop=True)
crank_angle = df["Crank_Angle"].values
n_pts = len(crank_angle)

# Selecionar ciclos uniformemente espaçados
pressure_cols = [c for c in df.columns if c.startswith("Cycle_")]
step          = max(1, len(pressure_cols) // MAX_CYCLES)
sel_cols      = pressure_cols[::step]
sel_labels    = [c.replace("Cycle_", "MP ") for c in sel_cols]
n_cycles      = len(sel_cols)

# Faixa global de pressão (para colormap consistente)
p_matrix       = df[sel_cols].values.astype(float)
z_min_global   = float(np.nanmin(p_matrix))
z_max_global   = float(np.nanmax(p_matrix))
z_baseline     = z_min_global * 0.98    # linha de base ligeiramente abaixo do mínimo

print(f"Ciclos no gráfico  : {n_cycles} (a cada {step})  →  {sel_labels[0]} ... {sel_labels[-1]}")
print(f"Pontos/ciclo       : {n_pts}  ({ANGLE_MIN}° a {ANGLE_MAX}°)")
print(f"Pressão global     : {z_min_global:.2f} a {z_max_global:.2f} bar")

Ciclos no gráfico  : 24 (a cada 4)  →  MP 74 ... MP 166
Pontos/ciclo       : 1201  (-30.0° a 90.0°)
Pressão global     : 1.75 a 71.88 bar


In [5]:
# ── 4. Construir o Waterfall 3D interativo ────────────────────────────────────
#
# Cada ciclo é um ribbon (superfície 2×N):
#   linha 0 → curva de pressão  (z = p(theta), y = i)
#   linha 1 → linha de base     (z = z_baseline, y = i)
# Cor mapeada pela pressão: azul=baixa, vermelho=alta (knock).

fig = go.Figure()

for i, (col, label) in enumerate(zip(sel_cols, sel_labels)):
    pressure = df[col].values.astype(float)

    # Geometria do ribbon (shape 2×N)
    x_surf = np.tile(crank_angle, (2, 1))                      # ângulo do virabrequim
    y_surf = np.full((2, n_pts), float(i))                     # profundidade = índice do ciclo
    z_surf = np.vstack([pressure, np.full(n_pts, z_baseline)]) # pressão (topo) + baseline
    c_surf = z_surf.copy()                                     # cor = valor de pressão

    is_last = (i == n_cycles - 1)

    fig.add_trace(
        go.Surface(
            x=x_surf,
            y=y_surf,
            z=z_surf,
            surfacecolor=c_surf,
            colorscale="RdYlBu_r",   # vermelho=alta, azul=baixa pressão
            cmin=z_min_global,
            cmax=z_max_global,
            showscale=is_last,       # barra de cor apenas no último ribbon
            colorbar=dict(
                title=dict(text="Pressão (bar)", side="right", font=dict(size=13)),
                thickness=16,
                len=0.65,
                x=1.02,
            ) if is_last else {},
            opacity=0.88,
            name=label,
            showlegend=False,
            hovertemplate=(
                f"<b>{label}</b><br>"
                "Ângulo: %{x:.1f}°<br>"
                "Pressão: %{z:.2f} bar"
                "<extra></extra>"
            ),
        )
    )

# Ticks do eixo Y: mostrar a cada 2 ciclos para não sobrecarregar
tick_step  = max(1, n_cycles // 10)
tick_vals  = list(range(0, n_cycles, tick_step))
tick_texts = [sel_labels[v] for v in tick_vals]

fig.update_layout(
    title=dict(
        text="Waterfall Plot — Pressão Cíclica de Combustão (H₂)",
        font=dict(size=20, family="Arial"),
        x=0.5,
        xanchor="center",
    ),
    scene=dict(
        xaxis=dict(
            title="Ângulo do Virabrequim (° aTDC)",
            title_font=dict(size=13),
            gridcolor="rgba(180,180,180,0.5)",
            showbackground=True,
            backgroundcolor="rgba(230,230,230,0.25)",
        ),
        yaxis=dict(
            title="Ciclo (MP)",
            title_font=dict(size=13),
            tickvals=tick_vals,
            ticktext=tick_texts,
            gridcolor="rgba(180,180,180,0.5)",
            showbackground=True,
            backgroundcolor="rgba(230,230,230,0.25)",
        ),
        zaxis=dict(
            title="Pressão (bar)",
            title_font=dict(size=13),
            gridcolor="rgba(180,180,180,0.5)",
            showbackground=True,
            backgroundcolor="rgba(230,230,230,0.25)",
        ),
        camera=dict(
            eye=dict(x=1.6, y=-1.9, z=0.75),
            up=dict(x=0, y=0, z=1),
        ),
        aspectratio=dict(x=1.5, y=1.0, z=0.7),
    ),
    margin=dict(l=0, r=90, t=80, b=0),
    width=1200,
    height=750,
    paper_bgcolor="white",
)

fig.show()
print(f"Gráfico renderizado com {n_cycles} ciclos.")

Gráfico renderizado com 24 ciclos.


In [ ]:
# ── 5. Salvar como HTML interativo ────────────────────────────────────────────
fig.write_html(
    str(HTML_OUTPUT),
    include_plotlyjs="cdn",   # referencia CDN → arquivo menor
    full_html=True,
)
print(f"Gráfico salvo em: {HTML_OUTPUT.resolve()}")